# GateKeep · notebook 06 · Task 10: verdict, plots, tau sweep, grader check

**Attach:** `gatekeep_data` and **notebook 04's output**. **Settings:** GPU T4 · Internet On · secret `OLLAMA_API_KEY`.
Needs the backup from notebook 05 (it holds the V0-V3 rows). Run steps 1-2 interactively, then **Save & Run All**. About 40-60 min.
**Paste back the whole output of steps 3, 4 and 5.**

## 1 · code, packages, key

In [ ]:
import glob, os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.isdir("GateKeep"):
    subprocess.run(["git", "-C", "GateKeep", "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/RishabhCodezZz/GateKeep.git"], check=True)
os.chdir("GateKeep")
os.environ["PYTHONPATH"] = "src"   # so `!python -m gatekeep.cli` finds the package
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" safetensors huggingface_hub scipy accelerate faiss-cpu sentence-transformers pymupdf pytest openai

In [ ]:
!nvidia-smi -L

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["OLLAMA_API_KEY"] = UserSecretsClient().get_secret("OLLAMA_API_KEY")

## 2 · restore (cache and data from the fullest backup, models from notebook 04's output)

In [ ]:
import sys
sys.path.insert(0, "src")
from gatekeep.restore import restore

COLD_START_OK = False   # set True only if you really want to rebuild everything (costs ~11% of the monthly quota)
where = restore("/kaggle/input", ".")
if where:
    print("restored from", where)
    subprocess.run(["python", "-m", "gatekeep.cli", "purge_empty"], check=True)  # drop empty replies cached before the thinking fix
elif not COLD_START_OK:
    raise SystemExit("STOPPED: no backup found in /kaggle/input. Attach your backup (the dataset must contain the folders "
                     "data/, cache/, results/ or a gatekeep_backup.zip), or set COLD_START_OK = True to rebuild from scratch.")
else:
    print("no backup: cold start, the book is re-parsed and every model call is repeated")
for name in ("review.csv", "handwritten.csv"):
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if found and not os.path.exists(name):
        shutil.copy(found[0], name)
    print(name, "->", "found" if os.path.exists(name) else "MISSING (add it to your dataset)")

In [ ]:
missing = [g for g in ("route", "grade", "grounded", "sufficient") if not os.path.exists(f"models/{g}/model.safetensors")]
assert not missing, (f"fine-tuned weights missing for {missing}. Attach notebook 04's OUTPUT as an input "
                     "(Add Input -> Notebook Outputs); it only has them if it was saved with Save & Run All. Otherwise re-run notebook 04.")
print("fine-tuned models found:", sorted(os.listdir("models")))

## 3 · verdict and plots (V1 vs V3)

In [ ]:
!python -m gatekeep.cli report

## 4 · threshold sweep: V3 at tau 0.7, 0.85, 0.95

In [ ]:
import json
for t in (0.7, 0.85, 0.95):
    f = f"results/tau_{t}.json"
    json.dump({g: t for g in ("route", "grade", "grounded", "sufficient")}, open(f, "w"))
    subprocess.run(["python", "-m", "gatekeep.cli", "variants", "V3", "", f, "0", f"tau{t}_"], check=True)

## 5 · is Super a valid grader? (Super vs Ultra agreement on 200 V1 answers, about 25 min)

In [ ]:
!python -m gatekeep.cli validate_grader

## 6 · back up, then download `gatekeep_backup.zip` from the Output panel into your dataset

In [ ]:
!cd /kaggle/working/GateKeep && zip -rq /kaggle/working/gatekeep_backup.zip data cache results models/ckpts.json && ls -lh /kaggle/working/gatekeep_backup.zip